# 02 — Logit Lens Analysis: Where Does Turkish Reasoning Diverge?

**Goal**: Apply the logit lens to Llama-3-8B to trace P(correct answer) across all 33 hidden state positions (embedding + 32 transformer layers). Compare English vs Turkish trajectories to identify WHERE reasoning diverges.

**Model**: Meta-Llama-3-8B (base model — standard for interpretability; Instruct fine-tuning adds confounds)

**Key technical detail**: We apply `model.model.norm` (RMSNorm) before `model.lm_head` to get meaningful logits. This is critical for Llama architectures.

In [ ]:
# Setup
import os
import sys
sys.path.insert(0, '..')

# os.environ['HF_TOKEN'] = 'your_token_here'

import numpy as np
import torch
from tqdm import tqdm

from src.utils import set_seed, setup_logging, get_gpu_info, cache_results, load_cached, ensure_dir
from src.model import LlamaWrapper
from src.data import load_matched_problems, construct_prompt
from src.logit_lens import compute_logit_lens, compute_js_divergence, detect_english_pivot
from src.metrics import (
    compute_divergence_onset, categorize_divergence_pattern,
    compute_layer_region_stats, aggregate_trajectories,
)

set_seed(42)
logger = setup_logging()
print(f"GPU: {get_gpu_info()}")
ensure_dir('../results/cached_activations')

In [ ]:
# Load base model (NOT Instruct — base is standard for interpretability)
wrapper = LlamaWrapper(
    model_name="meta-llama/Meta-Llama-3-8B",
    dtype="float16",
)
print(f"Model: {wrapper.n_layers} layers, hidden_dim={wrapper.hidden_dim}")

In [ ]:
# Sanity check: logit lens on a simple English completion
test_prompt = "The capital of France is"
test_result = compute_logit_lens(wrapper, test_prompt, target_token_ids=[], top_k=5)

print("Sanity check — top-1 predictions at each layer:")
for i, layer_top in enumerate(test_result['top_tokens']):
    if layer_top:
        token_str = layer_top[0][1].strip()
        prob = layer_top[0][2]
        print(f"  Layer {i:2d}: '{token_str}' (p={prob:.3f})")

print("\n✓ If 'Paris' appears in later layers with increasing probability, logit lens is working correctly.")

In [ ]:
# Load problems
problems = load_matched_problems(n_problems=30, seed=42)

# Show prompt format
p = problems[0]
print("English prompt:")
print(construct_prompt(p['question_en'], style='direct'))
print("\nTurkish prompt:")
print(construct_prompt(p['question_tr'], style='direct'))
print(f"\nTarget answer: {p['answer_number']}")
print(f"Target token IDs: {wrapper.get_answer_token_ids(p['answer_number'])}")

In [ ]:
# === CORE EXPERIMENT: Run logit lens on all 30 problems ===
# This is the main computation — takes ~30-60 min on A100

cache_path = '../results/cached_activations/logit_lens_results.pt'
cached = load_cached(cache_path)

if cached is not None:
    print("Loaded cached results.")
    all_results_en = cached['en']
    all_results_tr = cached['tr']
else:
    all_results_en = []
    all_results_tr = []

    for i, p in enumerate(tqdm(problems, desc="Logit lens")):
        target_ids = wrapper.get_answer_token_ids(p['answer_number'])

        # English
        prompt_en = construct_prompt(p['question_en'], style='direct')
        result_en = compute_logit_lens(wrapper, prompt_en, target_ids, top_k=10)
        result_en['prompt'] = prompt_en
        all_results_en.append(result_en)

        # Turkish
        prompt_tr = construct_prompt(p['question_tr'], style='direct')
        result_tr = compute_logit_lens(wrapper, prompt_tr, target_ids, top_k=10)
        result_tr['prompt'] = prompt_tr
        all_results_tr.append(result_tr)

        if (i + 1) % 5 == 0:
            print(f"  [{i+1}/{len(problems)}] EN final P={result_en['probs'][-1]:.4f}, TR final P={result_tr['probs'][-1]:.4f}")

    # Cache results
    cache_results({'en': all_results_en, 'tr': all_results_tr}, cache_path)
    print(f"Results cached to {cache_path}")

In [ ]:
# Extract probability trajectories
all_probs_en = [r['probs'] for r in all_results_en]
all_probs_tr = [r['probs'] for r in all_results_tr]

# Aggregate statistics
mean_en, ci_lo_en, ci_hi_en = aggregate_trajectories(all_probs_en)
mean_tr, ci_lo_tr, ci_hi_tr = aggregate_trajectories(all_probs_tr)

print("Mean P(answer) at final layer:")
print(f"  English: {mean_en[-1]:.4f}")
print(f"  Turkish: {mean_tr[-1]:.4f}")
print(f"  Ratio:   {mean_tr[-1] / max(mean_en[-1], 1e-10):.2f}x")

In [ ]:
# Quick trajectory visualization
from src.visualization import plot_aggregate_trajectory
import matplotlib.pyplot as plt

fig = plot_aggregate_trajectory(
    mean_en, ci_lo_en, ci_hi_en,
    mean_tr, ci_lo_tr, ci_hi_tr,
    save_path='../results/figures/fig2_aggregate_trajectory.pdf',
)
plt.show()
print("\n↑ This is the hero figure. If EN rises earlier/higher than TR, we have evidence.")

In [ ]:
# Divergence onset analysis
onset_layers = []
patterns = []

for probs_en, probs_tr, p in zip(all_probs_en, all_probs_tr, problems):
    onset = compute_divergence_onset(probs_en, probs_tr, threshold=0.1)
    pattern = categorize_divergence_pattern(probs_en, probs_tr)
    onset_layers.append(onset)
    patterns.append(pattern)

valid_onsets = [l for l in onset_layers if l >= 0]
print(f"Divergence onset statistics:")
print(f"  Problems with divergence: {len(valid_onsets)}/{len(onset_layers)}")
if valid_onsets:
    print(f"  Median onset layer: {np.median(valid_onsets):.0f}")
    print(f"  Mean onset layer: {np.mean(valid_onsets):.1f}")

print(f"\nDivergence patterns:")
from collections import Counter
for pattern, count in Counter(patterns).most_common():
    print(f"  {pattern}: {count} ({count/len(patterns)*100:.0f}%)")

In [ ]:
# Layer region analysis
region_stats = compute_layer_region_stats(all_probs_en, all_probs_tr)

print("Layer Region Analysis:")
print(f"{'Region':<10} {'Mean EN':>10} {'Mean TR':>10} {'Divergence':>12} {'Cohen d':>10} {'p-value':>10}")
print("-" * 65)
for region, stats in region_stats.items():
    print(f"{region:<10} {stats['mean_en']:>10.4f} {stats['mean_tr']:>10.4f} "
          f"{stats['mean_divergence']:>12.4f} {stats['cohens_d']:>10.2f} {stats['p_value']:>10.4f}")

In [ ]:
# Top-token analysis: what does the model predict at each layer?
# Check if Turkish prompts produce English predictions in mid-layers
print("\nTop-1 predictions for Turkish prompts at selected layers:")
print("(Testing the Wendler et al. 'latent English' hypothesis)\n")

for i in [0, 5, 10, 15, 20, 25, 30, 32]:
    if i >= len(all_results_tr[0]['top_tokens']):
        continue
    tokens_at_layer = []
    for result in all_results_tr:
        if result['top_tokens'][i]:
            tokens_at_layer.append(result['top_tokens'][i][0][1].strip())
    
    # Count ASCII vs non-ASCII (proxy for English vs Turkish)
    n_ascii = sum(1 for t in tokens_at_layer if all(ord(c) < 128 for c in t))
    print(f"  Layer {i:2d}: {n_ascii}/{len(tokens_at_layer)} ASCII (likely English) | Sample: {tokens_at_layer[:5]}")

In [ ]:
# JS Divergence across layers
all_js_div = []
for result_en, result_tr in zip(all_results_en, all_results_tr):
    js = compute_js_divergence(result_en['logits'], result_tr['logits'])
    all_js_div.append(js)

mean_js = np.mean(all_js_div, axis=0)
peak_layer = np.argmax(mean_js)
print(f"JS Divergence peak at layer {peak_layer} (value={mean_js[peak_layer]:.4f})")
print(f"This is where EN and TR processing strategies differ most.")

In [ ]:
# Entropy analysis
all_entropy_en = [r['entropy'] for r in all_results_en]
all_entropy_tr = [r['entropy'] for r in all_results_tr]

mean_entropy_en = np.mean(all_entropy_en, axis=0)
mean_entropy_tr = np.mean(all_entropy_tr, axis=0)

print("Mean entropy (bits) at selected layers:")
for i in [0, 10, 16, 20, 25, 32]:
    if i < len(mean_entropy_en):
        print(f"  Layer {i:2d}: EN={mean_entropy_en[i]:.2f} TR={mean_entropy_tr[i]:.2f} (diff={mean_entropy_tr[i]-mean_entropy_en[i]:+.2f})")

print(f"\nIf TR entropy is consistently higher in mid-layers, the model is 'confused' during Turkish reasoning.")

In [ ]:
# Save all results for visualization notebook
cache_results({
    'problems': problems,
    'all_probs_en': all_probs_en,
    'all_probs_tr': all_probs_tr,
    'mean_en': mean_en, 'ci_lo_en': ci_lo_en, 'ci_hi_en': ci_hi_en,
    'mean_tr': mean_tr, 'ci_lo_tr': ci_lo_tr, 'ci_hi_tr': ci_hi_tr,
    'onset_layers': onset_layers,
    'patterns': patterns,
    'region_stats': region_stats,
    'all_js_div': all_js_div,
    'mean_entropy_en': mean_entropy_en,
    'mean_entropy_tr': mean_entropy_tr,
    'all_results_en': all_results_en,
    'all_results_tr': all_results_tr,
}, '../results/cached_activations/analysis_results.pt')

print("All results saved for visualization notebook.")